## Analisis Topografico 
En estas celdas se carga el DEM y el directorio en el cual se va a guardar utilizando una optimizacion de C++ en el kernel de python. //
Se rellena el DEM, se calculan las redes de drenaje y las mascaras de las cuencas a partir de los outlets que se definen. 

In [ ]:
from TopoAnalysis import dem as d
import os
from pathlib import Path
import numpy as np
import rasterio
from rasterio.features import shapes
import geopandas as gpd
from shapely.geometry import shape

# ── Directory structure ──────────────────────────────────────────────────────

# Carpeta principal de trabajo
BASE_DIR = r"C:\datos\Memoria\Codigos Stanford"

# Todos los productos principales se guardarán aquí
DATA_DIR = r"C:\datos\Memoria\Codigos Stanford\Productos"

# Nombre del análisis
NB_NAME = 'dem_analysis'

# Subcarpetas dentro de Productos
RES_DIR  = DATA_DIR
MASK_DIR = os.path.join(DATA_DIR, 'masks')
POLY_DIR = os.path.join(DATA_DIR, 'basin_polygons')

# Crear carpetas si no existen
for folder in [DATA_DIR, MASK_DIR, POLY_DIR]:
    os.makedirs(folder, exist_ok=True)

print(f'Base     -> {BASE_DIR}')
print(f'Data     -> {DATA_DIR}')
print(f'Masks    -> {MASK_DIR}')
print(f'Polygons -> {POLY_DIR}')

In [ ]:
elevation = d.Elevation.load('C:/datos/Memoria/TDM_cut_UTM18.tif')
print(elevation._georef_info.xllcenter, elevation._georef_info.yllcenter)

In [ ]:
filled = d.FilledElevation(elevation = elevation)
print('Finished filled')

In [ ]:
flow_direction = d.FlowDirectionD8(flooded_dem = filled)
print('Finished flow direciton')
area = d.Area(flow_direction = flow_direction)
print('Finished area')
log_area = d.LogArea(area = area)
print('Finished log area')

In [ ]:
filled.save(os.path.join(DATA_DIR, 'rapel_dem_utm_filled30m'))
flow_direction.save(os.path.join(DATA_DIR, 'rapel_fd_utm30m'))
area.save(os.path.join(DATA_DIR, 'rapel_area_utm30m'))
log_area.save(os.path.join(DATA_DIR, 'rapel_logarea_utm30m'))

## Aqui se definen los outlets. 

In [ ]:
samples = [
    ['Rio Huenchullami', 769308, 6089892],
    ['Estero Cañete', 740807, 6092244],
    ['Estero Quebrada Honda', 728736, 6079032],
    ['Estero Malhueco', 729540, 6072948],
    ['Rio Pinotalca', 719340, 6060852],
    ['Rio Reloca', 717258, 6051540],
    ['Estero de Chanco', 720425, 6039864],
    ['Rio Rahue', 724668, 6037584],
    ['Rio Curanilahue', 719400, 6034812],
    ['Rio Curanipe', 713916, 6030936],
    ['Rio Chovellen', 714756, 6021096],
    ['Tregalemu', 703485, 6018819],
    ['Limite Maule-Ñuble', 702277, 6012996],
    ['Estero Buchupureo', 705000, 6006276],
    ['Cobquecura', 702961, 6000696],
    ['Rio Taucu', 701496, 5994768],
    ['Estero Colmuyado', 699396, 5979828],
    ['Estero Mela', 697332, 5976096]
]

print("listo")

In [ ]:
for sample, x, y in samples:
    try:
        basin_mask = d.Mask(flow_direction = flow_direction, outlets = ((x, y),))
        basin_mask.save(os.path.join(MASK_DIR, sample.replace('-','_') + '_mask'))

        print(f"Finished processing sample: {sample}")

    except Exception as e: 
        print(f"failed to process sample {sample}: {e}")

In [ ]:
for sample, _, _ in samples:
    basin_mask = d.Mask.load(os.path.join(MASK_DIR, sample.replace('-','_') + '_mask'))
    total_area = np.sum(basin_mask._griddata)*np.power(basin_mask._georef_info.dx,2)
    print(sample, float(total_area) / 1.0E6, "km2")

In [ ]:
# Masked rasters written above, one per sample
masked_rasters = [sample.replace('-','_') + '_mask' for sample, _, _ in samples]

# Specify the CRS (optional). Example: "EPSG:32719" for UTM Zone 19S
user_defined_crs = "EPSG:32718"  # Change or set to None to use the raster's CRS

# Loop through each masked raster and convert to polygon
for raster_file in masked_rasters:
    try:
        # Open the raster file
        with rasterio.open(os.path.join(MASK_DIR, raster_file)) as src:
            # Read the raster data
            image = src.read(1)  # Read the first band
            mask = image > 0  # Only include non-zero pixels
            
            # Convert raster to polygons
            results = (
                {"properties": {"value": value}, "geometry": shape(geom)}
                for geom, value in shapes(image, mask=mask, transform=src.transform)
            )
            
            # Convert to a GeoDataFrame
            gdf = gpd.GeoDataFrame.from_features(results, crs=src.crs)
            
            # Override CRS if user_defined_crs is provided
            if user_defined_crs:
                gdf = gdf.set_crs(user_defined_crs, allow_override=True)
                print(f"CRS set to: {user_defined_crs}")
            
            # Save as a shapefile
            output_file = os.path.join(POLY_DIR, raster_file + "_polygon.shp")
            gdf.to_file(output_file)
            print(f"Successfully converted {raster_file} to {output_file}")
    
    except Exception as e:
        print(f"Failed to process {raster_file}: {e}")